In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os

from PIL import Image, ImageEnhance
# from scipy.ndimage import convolve
from skimage.metrics import structural_similarity

from scripts.preprocessing import scale_range, crop_borders, get_best_rotation, histogram_equalization, unsharp_masking

# Initial Preprocessing Steps to make images easier to work with

In [25]:
# EDIT DIRECTORY VARIABLES AS NEEDED
# --- Main Directory: contains all folders/files
root = "S:/CheXpert/"
# --- This is the original root listed on the csv file paths
old_root = "CheXpert-v1.0/train/"
old_test_root = "test/"

# --- Input directory variables
source_train_root1 = f"{root}raw_data/CheXpert-v1.0 batch 2 (train 1)/"
source_train_root2 = f"{root}raw_data/CheXpert-v1.0 batch 3 (train 2)/"
source_train_root3 = f"{root}raw_data/CheXpert-v1.0 batch 4 (train 3)/"
source_valid_rad_root = f"{root}raw_data/CheXpert-v1.0 batch 1 (validate & csv)/valid/"
source_test_rad_root = f"{root}raw_data/test/"

# --- Output directory variables
train_valid_root = f"{root}train_valid/"
test_root = f"{root}test/"
valid_rad_root = f"{root}valid_rad/" 
test_rad_root = f"{root}test_rad/"

# --- Filepaths for dfs
train_df_filepath = f"{root}train_df.csv"
train_df_full_filepath = f"{root}train_df_full.csv"
#
valid_df_filepath = f"{root}valid_df.csv"
test_df_filepath = f"{root}test_df.csv"
#
valid_rad_df_filepath = f"{root}valid_rad_df.csv"
test_rad_df_filepath = f"{root}test_rad_df.csv"

# --- Image sizes
dims = [224, 384, 512]

In [26]:
# Preprocessing variables
# --- Value range for scaling image array
scale_min = 0
scale_max = 255
crop_q1_threshold, crop_q3_threshold = np.quantile([i for i in range(scale_min,scale_max)], [0.25, 0.75])

# --- Threshold for cropping borders (50% chance to cut off any borders)
cutoff = 0.5
threshold_range = (scale_max - scale_min) * 0.6

### Choose which datasets to process

Especially for the larger datasets, this preprocessing step will take a LONG time. Processing about 250 images takes about 1 minute. The full training set took over 10 hours to process.

The full training dataset is only necessary for EDA, so I do not recommend converting them.

In [48]:
# Choose the datasets to process
process_train_df = True
process_train_df_full = False ################ 
#
process_valid_df = True
process_test_df = True
#
process_valid_rad_df = True
process_test_rad_df = True

In [28]:
# Load the training/validation csvs
train_df = pd.read_csv(train_df_filepath)
train_df_full = pd.read_csv(train_df_full_filepath)
valid_df = pd.read_csv(valid_df_filepath)
test_df = pd.read_csv(test_df_filepath)
valid_rad_df = pd.read_csv(valid_rad_df_filepath)
test_rad_df = pd.read_csv(test_rad_df_filepath)

print(f"# rows in train_df: {len(train_df)}")
print(f"# rows in train_df_full: {len(train_df_full)}")
print(f"# rows in valid_df: {len(valid_df)}")
print(f"# rows in test_df: {len(test_df)}")
print(f"# rows in valid_rad_df: {len(valid_rad_df)}")
print(f"# rows in test_rad_df: {len(test_rad_df)}")

# rows in train_df: 46092
# rows in train_df_full: 121303
# rows in valid_df: 30353
# rows in test_df: 39371
# rows in valid_rad_df: 202
# rows in test_rad_df: 518


In [29]:
# def pipeline1(img_arr, kernel, he_sigma, scale_min, scale_max):
#     """Scale the range, sharpen by convolving with a kernel, then equalize the histogram"""
#     output = scale_range(img_arr, scale_min, scale_max)
#     output = convolve(output, kernel)
#     output = histogram_equalization(output, scale_min, scale_max, he_sigma)
#     return output

def pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max):
    """Scale the range, sharpen via unsharp masking, then equalize the histogram"""
    output = unsharp_masking(img_arr, usm_sigma, weight, scale_min, scale_max)
    output = histogram_equalization(output, scale_min, scale_max, he_sigma)
    return output

In [30]:
# Image enhancement variables
he_sigma = 5
usm_sigma = 10
weight = 1.2

## Preprocessing Steps:

* Scale the image values to the range [0-255]
<!-- * Crop out any border regions algorithmically -->
* Resize the training and validation images to (224x224), (384x284), (512x512)
<!-- * Find the 90-degree rotation which is closest to the average of a sample of 1000 x-ray images -->
* Convert the array to type uint8 for compatibility with Image
* Save the processed image as as jpeg file

### Preprocessing steps for enhanced images

* Scale the image values to the range [0-255]
<!-- * Crop out any border regions algorithmically -->
* Resize the training and validation images to (224x224), (384x284), (512x512)
<!-- * Find the 90-degree rotation which is closest to the average of a sample of 1000 x-ray images -->
* Sharpen the image using unsharp masking
* Equalize the histogram to increase contrast
* Convert the array to type uint8 for compatibility with Image
* Save the processed image as as jpeg file


In [31]:
os.path.exists(output_file_path2)

False

In [39]:
%%time
# Preprocessing steps for the valid rad set
if process_valid_rad_df:
    output_df = valid_rad_df.copy()
    n_instances = len(output_df)
    
    input_paths = output_df["source_file_path"]
    output_paths_list = [output_df[f"base{str(dim)}_file_path"] for dim in dims]
    output_paths_list2 = [output_df[f"base{str(dim)}_file_path"].str[:-4] + "_usm.jpg" for dim in dims]
    
    for output_paths,output_paths2,dim in zip(output_paths_list, output_paths_list2, dims):
        for i,(input_file_path, output_file_path, output_file_path2) in enumerate(zip(input_paths, output_paths, output_paths2), start=1):
            if i % 5000 == 0:
                print(f"Completed {i}/{n_instances}")
            # If processed image already exists, skip
            if os.path.exists(output_file_path2): 
                continue
            with Image.open(input_file_path) as img:
                img_arr = np.array(img)
                img_arr = np.array(Image.fromarray(img_arr).resize((dim, dim), resample=Image.Resampling.BILINEAR))
                img = Image.fromarray(scale_range(img_arr, scale_min, scale_max).astype(np.uint8))
                img.save(output_file_path, "JPEG", quality=95)
                #
                img_arr2 = pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max)
                img2 = Image.fromarray(img_arr2.astype(np.uint8))
                img2.save(output_file_path2, "JPEG", quality=95)


CPU times: total: 0 ns
Wall time: 7 ms


In [41]:
%%time
# Preprocessing steps for the test rad set
if process_test_rad_df:
    output_df = test_rad_df.copy()
    n_instances = len(output_df)
    
    input_paths = output_df["source_file_path"]
    output_paths_list = [output_df[f"base{str(dim)}_file_path"] for dim in dims]
    output_paths_list2 = [output_df[f"base{str(dim)}_file_path"].str[:-4] + "_usm.jpg" for dim in dims]
    
    for output_paths,output_paths2,dim in zip(output_paths_list, output_paths_list2, dims):
        for i,(input_file_path, output_file_path, output_file_path2) in enumerate(zip(input_paths, output_paths, output_paths2), start=1):
            if i % 5000 == 0:
                print(f"Completed {i}/{n_instances}")
            # If processed image already exists, skip
            if os.path.exists(output_file_path2): 
                continue
            with Image.open(input_file_path) as img:
                img_arr = np.array(img)
                img_arr = np.array(Image.fromarray(img_arr).resize((dim, dim), resample=Image.Resampling.BILINEAR))
                img = Image.fromarray(scale_range(img_arr, scale_min, scale_max).astype(np.uint8))
                img.save(output_file_path, "JPEG", quality=95)
                #
                img_arr2 = pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max)
                img2 = Image.fromarray(img_arr2.astype(np.uint8))
                img2.save(output_file_path2, "JPEG", quality=95)


CPU times: total: 0 ns
Wall time: 9.6 ms


In [43]:
%%time
# Preprocessing steps for the test set
if process_test_df:
    output_df = test_df.copy()
    n_instances = len(output_df)
    
    input_paths = output_df["source_file_path"]
    output_paths_list = [output_df[f"base{str(dim)}_file_path"] for dim in dims]
    output_paths_list2 = [output_df[f"base{str(dim)}_file_path"].str[:-4] + "_usm.jpg" for dim in dims]
    
    for output_paths,output_paths2,dim in zip(output_paths_list, output_paths_list2, dims):
        print(f"Processing for {dim}x{dim} images")
        for i,(input_file_path, output_file_path, output_file_path2) in enumerate(zip(input_paths, output_paths, output_paths2), start=1):
            if i % 5000 == 0:
                print(f"Completed {i}/{n_instances}")
            # If processed image already exists, skip
            if os.path.exists(output_file_path2): 
                continue
            with Image.open(input_file_path) as img:
                img_arr = np.array(img)
                img_arr = np.array(Image.fromarray(img_arr).resize((dim, dim), resample=Image.Resampling.BILINEAR))
                img = Image.fromarray(scale_range(img_arr, scale_min, scale_max).astype(np.uint8))
                img.save(output_file_path, "JPEG", quality=95)
                #
                img_arr2 = pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max)
                img2 = Image.fromarray(img_arr2.astype(np.uint8))
                img2.save(output_file_path2, "JPEG", quality=95)


Processing for 224x224 images
Completed 5000/39371
Completed 10000/39371
Completed 15000/39371
Completed 20000/39371
Completed 25000/39371
Completed 30000/39371
Completed 35000/39371
Processing for 384x384 images
Completed 5000/39371
Completed 10000/39371
Completed 15000/39371
Completed 20000/39371
Completed 25000/39371
Completed 30000/39371
Completed 35000/39371
Processing for 512x512 images
Completed 5000/39371
Completed 10000/39371
Completed 15000/39371
Completed 20000/39371
Completed 25000/39371
Completed 30000/39371
Completed 35000/39371
CPU times: total: 844 ms
Wall time: 849 ms


In [44]:
%%time
# Preprocessing steps for the test set
if process_valid_df:
    output_df = valid_df.copy()
    n_instances = len(output_df)
    
    input_paths = output_df["source_file_path"]
    output_paths_list = [output_df[f"base{str(dim)}_file_path"] for dim in dims]
    output_paths_list2 = [output_df[f"base{str(dim)}_file_path"].str[:-4] + "_usm.jpg" for dim in dims]
    
    for output_paths,output_paths2,dim in zip(output_paths_list, output_paths_list2, dims):
        print(f"Processing for {dim}x{dim} images")
        for i,(input_file_path, output_file_path, output_file_path2) in enumerate(zip(input_paths, output_paths, output_paths2), start=1):
            if i % 5000 == 0:
                print(f"Completed {i}/{n_instances}")
            # If processed image already exists, skip
            if os.path.exists(output_file_path2): 
                continue
            with Image.open(input_file_path) as img:
                img_arr = np.array(img)
                img_arr = np.array(Image.fromarray(img_arr).resize((dim, dim), resample=Image.Resampling.BILINEAR))
                img = Image.fromarray(scale_range(img_arr, scale_min, scale_max).astype(np.uint8))
                img.save(output_file_path, "JPEG", quality=95)
                #
                img_arr2 = pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max)
                img2 = Image.fromarray(img_arr2.astype(np.uint8))
                img2.save(output_file_path2, "JPEG", quality=95)

Processing for 224x224 images
Completed 5000/30353
Completed 10000/30353
Completed 15000/30353
Completed 20000/30353
Completed 25000/30353
Completed 30000/30353
Processing for 384x384 images
Completed 5000/30353
Completed 10000/30353
Completed 15000/30353
Completed 20000/30353
Completed 25000/30353
Completed 30000/30353
Processing for 512x512 images
Completed 5000/30353
Completed 10000/30353
Completed 15000/30353
Completed 20000/30353
Completed 25000/30353
Completed 30000/30353
CPU times: total: 734 ms
Wall time: 766 ms


In [49]:
%%time
# Preprocessing steps for the test set
if process_train_df_full:
    output_df = train_df_full.copy()
    n_instances = len(output_df)
    
    input_paths = output_df["source_file_path"]
    output_paths_list = [output_df[f"base{str(dim)}_file_path"] for dim in dims]
    output_paths_list2 = [output_df[f"base{str(dim)}_file_path"].str[:-4] + "_usm.jpg" for dim in dims]
    
    for output_paths,output_paths2,dim in zip(output_paths_list, output_paths_list2, dims):
        print(f"Processing for {dim}x{dim} images")
        for i,(input_file_path, output_file_path, output_file_path2) in enumerate(zip(input_paths, output_paths, output_paths2), start=1):
            if i % 5000 == 0:
                print(f"Completed {i}/{n_instances}")
            # If processed image already exists, skip
            if os.path.exists(output_file_path2): 
                continue
            with Image.open(input_file_path) as img:
                img_arr = np.array(img)
                img_arr = np.array(Image.fromarray(img_arr).resize((dim, dim), resample=Image.Resampling.BILINEAR))
                img = Image.fromarray(scale_range(img_arr, scale_min, scale_max).astype(np.uint8))
                img.save(output_file_path, "JPEG", quality=95)
                #
                img_arr2 = pipeline2(img_arr, weight, usm_sigma, he_sigma, scale_min, scale_max)
                img2 = Image.fromarray(img_arr2.astype(np.uint8))
                img2.save(output_file_path2, "JPEG", quality=95)

Processing for 224x224 images
Completed 5000/121303
Completed 10000/121303
Completed 15000/121303
Completed 20000/121303
Completed 25000/121303
Completed 30000/121303
Completed 35000/121303
Completed 40000/121303
Completed 45000/121303
Completed 50000/121303
Completed 55000/121303
Completed 60000/121303
Completed 65000/121303
Completed 70000/121303
Completed 75000/121303
Completed 80000/121303
Completed 85000/121303
Completed 90000/121303
Completed 95000/121303
Completed 100000/121303
Completed 105000/121303
Completed 110000/121303
Completed 115000/121303
Completed 120000/121303
Processing for 384x384 images
Completed 5000/121303
Completed 10000/121303
Completed 15000/121303
Completed 20000/121303
Completed 25000/121303
Completed 30000/121303
Completed 35000/121303
Completed 40000/121303
Completed 45000/121303
Completed 50000/121303
Completed 55000/121303
Completed 60000/121303
Completed 65000/121303
Completed 70000/121303
Completed 75000/121303
Completed 80000/121303
Completed 85000/1

In [19]:
# View the time estimate below to see how long this might take to run

# %%time
# # Preprocessing steps for the train set
# output_df = train_df.copy()

# for ind in output_df.index:
#     paths = list(output_df.loc[ind, ["source_file_path", 
#                                 "base224_file_path", "base224_file_path2", 
#                                 "base384_file_path", "base384_file_path2", 
#                                 "base512_file_path", "base512_file_path2"]])
#     with Image.open(paths[0]) as img:
#         img_arr = np.array(img)
#         img_arr_224 = np.array(Image.fromarray(img_arr).resize((224, 224), resample=Image.Resampling.BILINEAR))
#         img_arr_384 = np.array(Image.fromarray(img_arr).resize((384, 384), resample=Image.Resampling.BILINEAR))
#         img_arr_512 = np.array(Image.fromarray(img_arr).resize((512, 512), resample=Image.Resampling.BILINEAR))
#         #
#         img_224 = Image.fromarray(scale_range(img_arr_224, scale_min, scale_max).astype(np.uint8))
#         img_384 = Image.fromarray(scale_range(img_arr_384, scale_min, scale_max).astype(np.uint8))
#         img_512  = Image.fromarray(scale_range(img_arr_512, scale_min, scale_max).astype(np.uint8))
#         #
#         img_224.save(paths[1], "JPEG", quality=95)
#         img_384.save(paths[3], "JPEG", quality=95)
#         img_512.save(paths[5], "JPEG", quality=95)
#         ###
#         img_arr_224 = pipeline2(img_arr_224, weight, usm_sigma, he_sigma, scale_min, scale_max)
#         img_arr_384 = pipeline2(img_arr_384, weight, usm_sigma, he_sigma, scale_min, scale_max)
#         img_arr_512 = pipeline2(img_arr_512, weight, usm_sigma, he_sigma, scale_min, scale_max)
#         #
#         img_224 = Image.fromarray(scale_range(img_arr_224, scale_min, scale_max).astype(np.uint8))
#         img_384 = Image.fromarray(scale_range(img_arr_384, scale_min, scale_max).astype(np.uint8))
#         img_512  = Image.fromarray(scale_range(img_arr_512, scale_min, scale_max).astype(np.uint8))
#         #
#         img_224.save(paths[2], "JPEG", quality=95)
#         img_384.save(paths[4], "JPEG", quality=95)
#         img_512.save(paths[6], "JPEG", quality=95)

CPU times: total: 6h 21min 46s
Wall time: 10h 49min 9s
